In [1]:
import os, random
import numpy as np
import pandas as pd
import torch
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import roc_auc_score, log_loss, accuracy_score, average_precision_score, f1_score
from sentence_transformers import SentenceTransformer

from sklearn.model_selection import train_test_split

from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names
from deepctr_torch.models import DeepFM
from deepctr_torch.callbacks import EarlyStopping

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = 'cpu'

c:\Users\Casa\Desktop\U\10mo Semestre\Sistemas Recomendadores\Proyecto\Proyecto_RecSys\modelo_deepfm\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
games = pd.read_csv("Game Recommendations on Steam/games.csv")
recs = pd.read_csv("Game Recommendations on Steam/recommendations.csv")
desc = pd.read_csv("Steam Store Games/steam_description_data.csv")
media = pd.read_csv("Steam Store Games/steam_media_data.csv")
matched = pd.read_csv("games_steam_matched.csv")

In [3]:
desc = desc.rename(columns={"steam_appid": "appid"})
media = media.rename(columns={"steam_appid": "appid"})
matched = matched.merge(desc[['appid', 'about_the_game']], on='appid', how='left')
matched = matched.merge(media[['appid', 'header_image']], on='appid', how='left')
df_merged = matched 


In [4]:
df_merged.columns

Index(['app_id', 'title', 'date_release', 'win', 'mac', 'linux', 'rating',
       'positive_ratio', 'user_reviews', 'price_final', 'price_original',
       'discount', 'steam_deck', 'title_key', 'appid', 'name', 'release_date',
       'english', 'developer', 'publisher', 'platforms', 'required_age',
       'categories', 'genres', 'steamspy_tags', 'achievements',
       'positive_ratings', 'negative_ratings', 'average_playtime',
       'median_playtime', 'owners', 'price', 'name_key', 'about_the_game',
       'header_image'],
      dtype='object')

In [5]:
df_merged["game_title"] = df_merged["title"].fillna(df_merged["name"])
df_merged = df_merged.drop(columns=["title", "name"], errors="ignore")
# df_merged["header_image"] = df_merged["header_image_y"].fillna(df_merged["header_image_x"])
# df_merged = df_merged.drop(columns=["header_image_x", "header_image_y"], errors="ignore")
# df_merged["about_the_game"] = df_merged["about_the_game_y"].fillna(df_merged["about_the_game_x"])
# df_merged = df_merged.drop(columns=["about_the_game_x", "about_the_game_y"], errors="ignore")

print(df_merged.columns)

Index(['app_id', 'date_release', 'win', 'mac', 'linux', 'rating',
       'positive_ratio', 'user_reviews', 'price_final', 'price_original',
       'discount', 'steam_deck', 'title_key', 'appid', 'release_date',
       'english', 'developer', 'publisher', 'platforms', 'required_age',
       'categories', 'genres', 'steamspy_tags', 'achievements',
       'positive_ratings', 'negative_ratings', 'average_playtime',
       'median_playtime', 'owners', 'price', 'name_key', 'about_the_game',
       'header_image', 'game_title'],
      dtype='object')


In [7]:
df_merged.head()

,app_id,date_release,win,mac,linux,rating,positive_ratio,user_reviews,price_final,price_original,...,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price,name_key,about_the_game,header_image,game_title
0,13500,2008-11-21,True,False,False,Very Positive,84,2199,9.99,9.99,...,657,101,307,312,200000-500000,8.59,prince of persia warrior withintm,<p>Enter the dark underworld of Prince of Pers...,https://steamcdn-a.akamaihd.net/steam/apps/135...,Prince of Persia: Warrior Within™
1,113020,2013-04-24,True,True,True,Very Positive,92,3722,14.99,14.99,...,7237,749,183,135,2000000-5000000,10.99,monaco what s yours is mine,Monaco: What's Yours Is Mine is a single playe...,https://steamcdn-a.akamaihd.net/steam/apps/113...,Monaco: What's Yours Is Mine
2,226560,2014-11-18,True,False,False,Mixed,61,873,14.99,14.99,...,549,383,0,0,50000-100000,10.99,escape dead island,"<img src=""https://steamcdn-a.akamaihd.net/stea...",https://steamcdn-a.akamaihd.net/steam/apps/226...,Escape Dead Island
3,249050,2014-10-27,True,True,False,Very Positive,88,8784,11.99,11.99,...,7423,1046,648,295,500000-1000000,8.99,dungeon of the endlesstm,"<img src=""https://steamcdn-a.akamaihd.net/stea...",https://steamcdn-a.akamaihd.net/steam/apps/249...,Dungeon of the ENDLESS™
4,250180,2015-09-14,True,False,False,Very Positive,90,5579,7.99,7.99,...,5229,553,212,242,500000-1000000,5.99,metal slug 3,"<i>“METAL SLUG 3”</i>, the masterpiece in SNK’...",https://steamcdn-a.akamaihd.net/steam/apps/250...,METAL SLUG 3


In [6]:
data = 3
if data==1:
    train_split = pd.read_csv("data/split/train_split.csv")
    val_split   = pd.read_csv("data/split/val_split.csv")
    test_split  = pd.read_csv("data/split/test_split.csv")
    sampled_recs = pd.read_csv("data/sampled_recommendations.csv")
elif data == 3:
    train_split = pd.read_csv("data3/split/train_split.csv")
    val_split   = pd.read_csv("data3/split/val_split.csv")
    test_split  = pd.read_csv("data3/split/test_split.csv")
    sampled_recs = pd.read_csv("data3/sampled_recommendations.csv")

In [7]:
train_merged = train_split.merge(df_merged, on="app_id", how="left")
val_merged   = val_split.merge(df_merged, on="app_id", how="left")
test_merged  = test_split.merge(df_merged, on="app_id", how="left")
sampled_recs_merged  = sampled_recs.merge(df_merged, on="app_id", how="left")

In [8]:
for name, df in [("train", train_merged), ("val", val_merged), ("test", test_merged)]:
    print(name, "positivos:", df["is_recommended"].sum(), "total:", len(df),
          "ratio:", df["is_recommended"].mean())

train positivos: 55757 total: 63905 ratio: 0.8724982395743682
val positivos: 9977 total: 11649 ratio: 0.8564683663833805
test positivos: 10888 total: 12784 ratio: 0.8516896120150188


In [9]:
test_user_pos = test_merged.groupby("user_id")["is_recommended"].sum()
print("Positivos por usuario (test):")
print(test_user_pos.describe())
print("Usuarios sin positivos en test:", (test_user_pos == 0).sum())

Positivos por usuario (test):
count    9906.000000
mean        1.099132
std         0.922688
min         0.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        28.000000
Name: is_recommended, dtype: float64
Usuarios sin positivos en test: 1228


In [13]:
sampled_recs_merged.columns

Index(['app_id', 'helpful', 'funny', 'date', 'is_recommended', 'hours',
       'user_id', 'review_id', 'date_release', 'win', 'mac', 'linux', 'rating',
       'positive_ratio', 'user_reviews', 'price_final', 'price_original',
       'discount', 'steam_deck', 'title_key', 'appid', 'release_date',
       'english', 'developer', 'publisher', 'platforms', 'required_age',
       'categories', 'genres', 'steamspy_tags', 'achievements',
       'positive_ratings', 'negative_ratings', 'average_playtime',
       'median_playtime', 'owners', 'price', 'name_key', 'about_the_game',
       'header_image', 'game_title'],
      dtype='object')

In [14]:
sampled_recs_merged.head()

,app_id,helpful,funny,date,is_recommended,hours,user_id,review_id,date_release,win,...,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price,name_key,about_the_game,header_image,game_title
0,322330,0,0,2019-07-02,True,67.5,731,33484606,2016-04-21,True,...,108601.0,4206.0,1070.0,404.0,5000000-10000000,10.99,don t starve together,Don't Starve Together is the standalone multip...,https://steamcdn-a.akamaihd.net/steam/apps/322...,Don't Starve Together
1,433340,0,0,2020-01-24,True,32.3,731,26236770,2017-08-01,True,...,24205.0,965.0,1314.0,827.0,1000000-2000000,14.99,slime rancher,"<img src=""https://steamcdn-a.akamaihd.net/stea...",https://steamcdn-a.akamaihd.net/steam/apps/433...,Slime Rancher
2,394360,2,0,2020-04-20,True,403.7,731,25992499,2016-06-06,True,...,34711.0,5251.0,9413.0,4181.0,1000000-2000000,34.99,hearts of iron iv,<strong>Victory is at your fingertips!</strong...,https://steamcdn-a.akamaihd.net/steam/apps/394...,Hearts of Iron IV
3,4700,0,0,2020-04-21,True,683.5,731,9845461,2007-11-27,True,...,10270.0,605.0,469.0,361.0,2000000-5000000,0.00,total war medieval ii definitive edition,"<img src=""https://steamcdn-a.akamaihd.net/stea...",https://steamcdn-a.akamaihd.net/steam/apps/470...,Total War: MEDIEVAL II – Definitive Edition
4,255710,0,0,2020-05-02,True,37.6,731,33641355,2015-03-10,True,...,67553.0,6005.0,3225.0,444.0,5000000-10000000,22.99,cities skylines,<strong>Cities: Skylines</strong> is a modern ...,https://steamcdn-a.akamaihd.net/steam/apps/255...,Cities: Skylines


In [10]:
mask_nan_genres = sampled_recs_merged["genres"].isna()

users_with_nan_genres = sampled_recs_merged.loc[mask_nan_genres, "user_id"].nunique()
rows_with_nan_genres = mask_nan_genres.sum()

print("Filas con genres NaN en sampled_recs_merged:", rows_with_nan_genres)
print("Usuarios distintos con al menos un genres NaN:", users_with_nan_genres)

Filas con genres NaN en sampled_recs_merged: 29135
Usuarios distintos con al menos un genres NaN: 8660


In [75]:
sampled_recs_merged["rating"].head()

0    5
1    5
2    6
3    5
4    6
Name: rating, dtype: int64

In [11]:
target = ['is_recommended']

sparse_features = [
    'user_id',   
    'rating',
]

dense_features = [
    'price_final',
    'average_playtime',
]

In [12]:
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
import numpy as np

sampled_recs_merged[sparse_features] = sampled_recs_merged[sparse_features].fillna('-1')
sampled_recs_merged[dense_features] = sampled_recs_merged[dense_features].fillna(0)

for feat in sparse_features:
    lbe = LabelEncoder()
    sampled_recs_merged[feat] = lbe.fit_transform(sampled_recs_merged[feat].astype(str))

mms = MinMaxScaler(feature_range=(0, 1))
sampled_recs_merged[dense_features] = mms.fit_transform(sampled_recs_merged[dense_features])

In [13]:
sampled_recs_merged["text_input"] = (
    sampled_recs_merged["about_the_game"]
    .fillna(sampled_recs_merged["game_title"])
    .fillna("")
)

Embedding con All MiniLM-L6-v2

In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np

ejecutar_embedding = False
if ejecutar_embedding:
    text_model = SentenceTransformer("all-MiniLM-L6-v2")  

    texts = sampled_recs_merged["text_input"].tolist()

    text_emb = text_model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    emb_dim = text_emb.shape[1]
    print("Dimensión embedding de texto:", emb_dim)


Embedding de texto con all-mpnet-base-v2


In [35]:
ejecutar_embedding = True
if ejecutar_embedding:
    text_model = SentenceTransformer('all-mpnet-base-v2') 

    texts = sampled_recs_merged["text_input"].tolist()

    text_emb = text_model.encode(
        texts,
        batch_size=64,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    emb_dim = text_emb.shape[1]
    print("Dimensión embedding de texto:", emb_dim)

c:\Users\Casa\Desktop\U\10mo Semestre\Sistemas Recomendadores\Proyecto\Proyecto_RecSys\modelo_deepfm\venv\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Casa\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled f

Dimensión embedding de texto: 768


In [36]:
np.save("text_emb_all-mpnet-base-v2.npy", text_emb)

In [20]:
# # Guardar
# # import numpy as np

# np.save("text_emb_sampled.npy", text_emb)
# np.save("text_emb_sampled3.npy", text_emb)

In [15]:
# # Cargar
# import numpy as np
data3 = 3
embedding = "mpnet"
if data ==1:
    text_emb = np.load("text_emb_sampled.npy")
    print(text_emb.shape)
    emb_dim = text_emb.shape[1]
elif data ==3:
    if embedding == "mpnet":
        text_emb = np.load("text_emb_all-mpnet-base-v2.npy")
    else:
        text_emb = np.load("text_emb_sampled3.npy")
        print(text_emb.shape)
    emb_dim = text_emb.shape[1]
    print(text_emb.shape)

(88494, 768)


In [ ]:

import clip
import torch
from PIL import Image
import requests
from io import BytesIO
import numpy as np
from tqdm import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_clip, preprocess = clip.load("ViT-B/32", device=device)

def get_image_embedding(url):
    try:
        if not isinstance(url, str) or url.strip() == "":
            raise ValueError("empty url")
        resp = requests.get(url, timeout=5)
        resp.raise_for_status()
        image = Image.open(BytesIO(resp.content)).convert("RGB")
        img = preprocess(image).unsqueeze(0).to(device)

        with torch.no_grad():
            emb = model_clip.encode_image(img)

        emb = emb / emb.norm(dim=-1, keepdim=True) 
        return emb.cpu().numpy().flatten()
    except Exception:
        return np.zeros((512,), dtype="float32")  



Device: cpu


In [23]:
guardar_embedding_img = False   
if guardar_embedding_img: 
    img_emb_list = []

    for url in tqdm(sampled_recs_merged["header_image"], desc="Imagenes"):
        emb = get_image_embedding(url)
        img_emb_list.append(emb)

    img_emb = np.vstack(img_emb_list).astype("float32")  
    img_emb_dim = img_emb.shape[1]
    print("Shape img_emb:", img_emb.shape)

    np.save("clip_img_emb_by_app.npy", img_emb)



In [38]:
img_emb = np.load("clip_img_emb_by_app.npy")
img_emb_dim = img_emb.shape[1]
print("Shape img_emb:", img_emb.shape)

Shape img_emb: (88494, 512)


In [39]:
appid2idx = {app_id: i for i, app_id in enumerate(sampled_recs_merged["app_id"])}

In [40]:
def get_split_img_emb(split_df, appid2idx, img_emb):
    idxs = [appid2idx[aid] for aid in split_df["app_id"]]
    return img_emb[idxs]

train_img_emb = get_split_img_emb(train_merged, appid2idx, img_emb)
val_img_emb   = get_split_img_emb(val_merged, appid2idx, img_emb)
test_img_emb  = get_split_img_emb(test_merged, appid2idx, img_emb)

print(train_img_emb.shape, val_img_emb.shape, test_img_emb.shape)


(63905, 512) (11649, 512) (12784, 512)


In [27]:
'''
from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names
fixlen_feature_columns = (
    [SparseFeat(feat, vocabulary_size=sampled_recs_merged[feat].nunique(), embedding_dim=8)
     for feat in sparse_features]
    +
    [DenseFeat(feat, 1) for feat in dense_features]
    +
    [DenseFeat("text_emb", emb_dim)]  # embedding de texto
)
dnn_feature_columns = fixlen_feature_columns
linear_feature_columns = fixlen_feature_columns
feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)
'''


'\nfrom deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names\nfixlen_feature_columns = (\n    [SparseFeat(feat, vocabulary_size=sampled_recs_merged[feat].nunique(), embedding_dim=8)\n     for feat in sparse_features]\n    +\n    [DenseFeat(feat, 1) for feat in dense_features]\n    +\n    [DenseFeat("text_emb", emb_dim)]  # embedding de texto\n)\ndnn_feature_columns = fixlen_feature_columns\nlinear_feature_columns = fixlen_feature_columns\nfeature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)\n'

In [41]:
from deepctr_torch.inputs import SparseFeat, DenseFeat, get_feature_names

fixlen_feature_columns = (
    [
        SparseFeat(
            feat,
            vocabulary_size=sampled_recs_merged[feat].nunique(),
            embedding_dim=8
        )
        for feat in sparse_features
    ]
    +
    [DenseFeat(feat, 1) for feat in dense_features]
    +
    [DenseFeat("text_emb", emb_dim)]      
    +
    [DenseFeat("img_emb", img_emb_dim)]   
)

dnn_feature_columns = fixlen_feature_columns
linear_feature_columns = fixlen_feature_columns

feature_names = get_feature_names(linear_feature_columns + dnn_feature_columns)


In [20]:
id2idx = {rid: i for i, rid in enumerate(sampled_recs_merged["review_id"])}
emb_dim = text_emb.shape[1]
print("Dim embedding:", emb_dim)

Dim embedding: 768


In [21]:
import numpy as np

def get_split_text_emb(split_df, id2idx, text_emb):
    idxs = [id2idx[rid] for rid in split_df["review_id"]]
    return text_emb[idxs]

train_text_emb = get_split_text_emb(train_merged, id2idx, text_emb)
val_text_emb   = get_split_text_emb(val_merged, id2idx, text_emb)
test_text_emb  = get_split_text_emb(test_merged, id2idx, text_emb)

print(train_text_emb.shape, val_text_emb.shape, test_text_emb.shape)


(63905, 768) (11649, 768) (12784, 768)


In [22]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

train_merged[sparse_features] = train_merged[sparse_features].fillna('-1')
val_merged[sparse_features]   = val_merged[sparse_features].fillna('-1')
test_merged[sparse_features]  = test_merged[sparse_features].fillna('-1')

for feat in sparse_features:
    lbe = LabelEncoder()
    
    all_vals = pd.concat([
        train_merged[feat],
        val_merged[feat],
        test_merged[feat]
    ], axis=0).astype(str)
    
    lbe.fit(all_vals)
    
    train_merged[feat] = lbe.transform(train_merged[feat].astype(str))
    val_merged[feat]   = lbe.transform(val_merged[feat].astype(str))
    test_merged[feat]  = lbe.transform(test_merged[feat].astype(str))


In [23]:
import numpy as np

for feat in dense_features:
    train_merged[feat] = pd.to_numeric(train_merged[feat], errors='coerce')
    val_merged[feat]   = pd.to_numeric(val_merged[feat], errors='coerce')
    test_merged[feat]  = pd.to_numeric(test_merged[feat], errors='coerce')

    train_merged[feat] = train_merged[feat].fillna(0)
    val_merged[feat]   = val_merged[feat].fillna(0)
    test_merged[feat]  = test_merged[feat].fillna(0)


In [24]:
for df in [train_merged, val_merged, test_merged]:
    df["is_recommended"] = df["is_recommended"].astype(int)

In [25]:
for feat in sparse_features:
    train_merged[feat] = train_merged[feat].astype('int32')
    val_merged[feat]   = val_merged[feat].astype('int32')
    test_merged[feat]  = test_merged[feat].astype('int32')

for feat in dense_features:
    train_merged[feat] = train_merged[feat].astype('float32')
    val_merged[feat]   = val_merged[feat].astype('float32')
    test_merged[feat]  = test_merged[feat].astype('float32')

train_text_emb = train_text_emb.astype('float32')
val_text_emb   = val_text_emb.astype('float32')
test_text_emb  = test_text_emb.astype('float32')

y_train = train_merged[target].values.astype('float32')
y_val   = val_merged[target].values.astype('float32')
y_test  = test_merged[target].values.astype('float32')


In [26]:
train_model_input = {
    name: train_merged[name].values
    for name in feature_names
    if name not in ["text_emb", "img_emb"]
}
train_model_input["text_emb"] = train_text_emb.astype("float32")
train_model_input["img_emb"]  = train_img_emb.astype("float32")

val_model_input = {
    name: val_merged[name].values
    for name in feature_names
    if name not in ["text_emb", "img_emb"]
}
val_model_input["text_emb"] = val_text_emb.astype("float32")
val_model_input["img_emb"]  = val_img_emb.astype("float32")

test_model_input = {
    name: test_merged[name].values
    for name in feature_names
    if name not in ["text_emb", "img_emb"]
}
test_model_input["text_emb"] = test_text_emb.astype("float32")
test_model_input["img_emb"]  = test_img_emb.astype("float32")


In [27]:
import torch
from deepctr_torch.models import DeepFM

device = "cuda" if torch.cuda.is_available() else "cpu"

model = DeepFM(
    linear_feature_columns=linear_feature_columns,
    dnn_feature_columns=dnn_feature_columns,
    task="binary",
    device=device,
)

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["auc"],
)

model.fit(
    train_model_input,
    y_train,
    batch_size=256,
    epochs=10,
    verbose=2,
    validation_data=(val_model_input, y_val),
)

pred_test = model.predict(test_model_input, batch_size=256)


cpu
Train on 63905 samples, validate on 11649 samples, 250 steps per epoch
Epoch 1/10
3s - loss:  0.4713 - auc:  0.5644 - val_auc:  0.6844
Epoch 2/10
3s - loss:  0.3642 - auc:  0.7363 - val_auc:  0.6969
Epoch 3/10
3s - loss:  0.3664 - auc:  0.8044 - val_auc:  0.7238
Epoch 4/10
3s - loss:  0.2950 - auc:  0.8572 - val_auc:  0.7243
Epoch 5/10
3s - loss:  0.2933 - auc:  0.8694 - val_auc:  0.7103
Epoch 6/10
3s - loss:  0.2766 - auc:  0.8785 - val_auc:  0.7172
Epoch 7/10
3s - loss:  0.2678 - auc:  0.8884 - val_auc:  0.7110
Epoch 8/10
3s - loss:  0.2519 - auc:  0.8963 - val_auc:  0.7162
Epoch 9/10
4s - loss:  0.2458 - auc:  0.9021 - val_auc:  0.6945
Epoch 10/10
3s - loss:  0.2474 - auc:  0.9035 - val_auc:  0.7131


In [28]:
from sklearn.metrics import roc_auc_score, log_loss

print("AUC test:", roc_auc_score(y_test, pred_test))
print("LogLoss test:", log_loss(y_test, pred_test))

AUC test: 0.7071806058696601
LogLoss test: 0.6125005598326378


In [29]:
def build_user_seen(train_df, val_df):
    seen = {}
    for df in [train_df, val_df]:
        for u, group in df.groupby("user_id"):
            items_u = set(group["app_id"].tolist())
            if u in seen:
                seen[u].update(items_u)
            else:
                seen[u] = items_u
    return seen

user_seen = build_user_seen(train_merged, val_merged)
len(user_seen), list(user_seen.items())[:3]

(9906,
 [(0, {244850, 261470, 294100}),
  (1, {269210, 322330, 444090, 674940}),
  (2,
   {214950,
    230410,
    306130,
    485510,
    582660,
    779340,
    1272320,
    1325200,
    1578650})])

In [54]:
def build_candidates_rank_df(model, test_df, user_seen, all_items,
                             id2idx, text_emb, feature_names,
                             appid2idx, img_emb, K=10, max_candidates=500):
    rows = []

    for u, df_u in test_df.groupby("user_id"):
        seen_u = user_seen.get(u, set())
        pos_items = set(df_u[df_u["is_recommended"] == 1]["app_id"].tolist())

        candidates = np.setdiff1d(all_items, np.fromiter(seen_u, dtype=int), assume_unique=True)

        if len(candidates) == 0 or len(pos_items) == 0:
            continue

        if max_candidates is not None and len(candidates) > max_candidates:
            rng = np.random.default_rng(42)
            sampled_neg = set(rng.choice(candidates, size=max_candidates, replace=False))
        else:
            sampled_neg = set(candidates)

        items_eval = list(pos_items.union(sampled_neg))

        df_eval = pd.DataFrame({
            "user_id": [u] * len(items_eval),
            "app_id": items_eval,
        })

        df_eval = df_eval.merge(
            test_df[
                ["user_id", "app_id", "rating", "price_final", "average_playtime", "is_recommended", "review_id"]
                + (["genres"] if "genres" in test_df.columns else [])
            ],
            on=["user_id", "app_id"],
            how="left"
        )

        df_eval["is_recommended"] = df_eval["is_recommended"].fillna(0).astype(int)
        df_eval["rating"] = df_eval["rating"].fillna(0).astype("int32")
        df_eval["price_final"] = df_eval["price_final"].fillna(0).astype("float32")
        df_eval["average_playtime"] = df_eval["average_playtime"].fillna(0).astype("float32")
        df_eval["user_id"] = df_eval["user_id"].astype("int32")

        if "review_id" in df_eval.columns:
            idxs = []
            for rid in df_eval["review_id"]:
                if pd.isna(rid) or rid not in id2idx:
                    idxs.append(None)
                else:
                    idxs.append(id2idx[rid])

            text_emb_local = np.zeros((len(df_eval), text_emb.shape[1]), dtype=text_emb.dtype)
            valid_idx = [i for i, idx in enumerate(idxs) if idx is not None]
            if valid_idx:
                text_emb_local[valid_idx] = text_emb[
                    [id2idx[df_eval["review_id"].iloc[i]] for i in valid_idx]
                ]
        else:
            text_emb_local = np.zeros((len(df_eval), text_emb.shape[1]), dtype=text_emb.dtype)


        if img_emb is not None:
            img_idxs = []
            for aid in df_eval["app_id"]:
                if aid in appid2idx:
                    img_idxs.append(appid2idx[aid])
                else:
                    img_idxs.append(None)

            img_emb_local = np.zeros((len(df_eval), img_emb.shape[1]), dtype=img_emb.dtype)
            valid_img_idx = [i for i, idx in enumerate(img_idxs) if idx is not None]
            if valid_img_idx:
                img_emb_local[valid_img_idx] = img_emb[
                    [appid2idx[df_eval["app_id"].iloc[i]] for i in valid_img_idx]
                ]
        else:
            print("No image embeddings provided.")
            img_emb_local = None

        for feat in dense_features:
            df_eval[feat] = df_eval[feat].astype("float32")

        X_rank = {
            name: df_eval[name].values
            for name in feature_names
            if name not in ["text_emb", "img_emb"]  
        }
        X_rank["text_emb"] = text_emb_local
        if img_emb_local is not None:
            X_rank["img_emb"] = img_emb_local

        scores = model.predict(X_rank, batch_size=256).reshape(-1)

        tmp = pd.DataFrame({
            "user_id": df_eval["user_id"].values,
            "app_id": df_eval["app_id"].values,
            "label": df_eval["is_recommended"].astype(int).values,
            "score": scores,
        })
        if "genres" in df_eval.columns:
            tmp["genres"] = df_eval["genres"].values

        rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=["user_id", "app_id", "label", "score"])

    rank_df = pd.concat(rows, ignore_index=True)
    rank_df = rank_df.sort_values(["user_id", "score"], ascending=[True, False]).reset_index(drop=True)
    return rank_df


all_items = df_merged["app_id"].unique()  


In [57]:
rank_df_pure = build_candidates_rank_df(
    model=model,
    test_df=test_merged,
    user_seen=user_seen,
    all_items=all_items,
    id2idx=id2idx,         
    text_emb=text_emb,      
    feature_names=feature_names,
    appid2idx=appid2idx,    
    img_emb=img_emb,        
    K=10,
    max_candidates=None,
)

rank_df_pure.head()

,user_id,app_id,label,score,genres
0,1,632360,1,0.838192,Action;Indie;Early Access
1,1,896160,0,0.544191,NaN
2,1,1700,0,0.534311,NaN
3,1,369070,0,0.522670,NaN
4,1,337340,0,0.518890,NaN


In [30]:
# Crea el catálogo único de ítems con sus características reales
item_cols = ["app_id", "price_final", "average_playtime"]
if "genres" in df_merged.columns:
    item_cols.append("genres")

# "El Menú": una ficha por juego
catalog_df = df_merged[item_cols].drop_duplicates(subset=["app_id"]).copy()

In [31]:
def build_candidates_rank_df(model, test_df, user_seen, all_items, 
                             catalog_df,  # <--- NUEVO ARGUMENTO AQUÍ
                             id2idx, text_emb, feature_names,
                             appid2idx, img_emb, dense_features, # Añadí dense_features a los args por seguridad
                             K=10, max_candidates=500):
    rows = []

    for u, df_u in test_df.groupby("user_id"):
        seen_u = user_seen.get(u, set())
        pos_items = set(df_u[df_u["is_recommended"] == 1]["app_id"].tolist())

        candidates = np.setdiff1d(all_items, np.fromiter(seen_u, dtype=int), assume_unique=True)

        if len(candidates) == 0 or len(pos_items) == 0:
            continue

        # Sampling de negativos
        if max_candidates is not None and len(candidates) > max_candidates:
            rng = np.random.default_rng(42)
            sampled_neg = set(rng.choice(candidates, size=max_candidates, replace=False))
        else:
            sampled_neg = set(candidates)

        items_eval = list(pos_items.union(sampled_neg))

        df_eval = pd.DataFrame({
            "user_id": [u] * len(items_eval),
            "app_id": items_eval,
        })

        # --- INICIO CORRECCIÓN ---
        
        # 1. Merge de INTERACCIÓN (test_df): Trae labels, rating del usuario y review específica
        df_eval = df_eval.merge(
            test_df[["user_id", "app_id", "is_recommended", "rating", "review_id"]],
            on=["user_id", "app_id"],
            how="left"
        )

        # 2. Merge de CATÁLOGO (catalog_df): Trae precio, playtime y géneros REALES para todos
        df_eval = df_eval.merge(
            catalog_df,
            on="app_id",
            how="left"
        )
        
        # --- FIN CORRECCIÓN ---

        # Rellenar nulos
        df_eval["is_recommended"] = df_eval["is_recommended"].fillna(0).astype(int)
        df_eval["rating"] = df_eval["rating"].fillna(0).astype("int32") # Correcto que sea 0 en negativos
        
        # Ahora price y playtime vienen del catálogo, así que los negativos tendrán valor real.
        # El fillna(0) queda solo por seguridad si el catálogo tuviera huecos.
        df_eval["price_final"] = df_eval["price_final"].fillna(0).astype("float32")
        df_eval["average_playtime"] = df_eval["average_playtime"].fillna(0).astype("float32")
        df_eval["user_id"] = df_eval["user_id"].astype("int32")

        # --- Lógica de Texto (Sin cambios) ---
        if "review_id" in df_eval.columns:
            idxs = []
            for rid in df_eval["review_id"]:
                if pd.isna(rid) or rid not in id2idx:
                    idxs.append(None)
                else:
                    idxs.append(id2idx[rid])

            text_emb_local = np.zeros((len(df_eval), text_emb.shape[1]), dtype=text_emb.dtype)
            valid_idx = [i for i, idx in enumerate(idxs) if idx is not None]
            if valid_idx:
                text_emb_local[valid_idx] = text_emb[
                    [id2idx[df_eval["review_id"].iloc[i]] for i in valid_idx]
                ]
        else:
            text_emb_local = np.zeros((len(df_eval), text_emb.shape[1]), dtype=text_emb.dtype)

        # --- Lógica de Imágenes (Sin cambios) ---
        if img_emb is not None:
            img_idxs = []
            for aid in df_eval["app_id"]:
                if aid in appid2idx:
                    img_idxs.append(appid2idx[aid])
                else:
                    img_idxs.append(None)

            img_emb_local = np.zeros((len(df_eval), img_emb.shape[1]), dtype=img_emb.dtype)
            valid_img_idx = [i for i, idx in enumerate(img_idxs) if idx is not None]
            if valid_img_idx:
                img_emb_local[valid_img_idx] = img_emb[
                    [appid2idx[df_eval["app_id"].iloc[i]] for i in valid_img_idx]
                ]
        else:
            # print("No image embeddings provided.") # Comentado para no ensuciar el output
            img_emb_local = None

        for feat in dense_features:
            if feat in df_eval.columns:
                df_eval[feat] = df_eval[feat].astype("float32")

        X_rank = {
            name: df_eval[name].values
            for name in feature_names
            if name not in ["text_emb", "img_emb"]  
        }
        X_rank["text_emb"] = text_emb_local
        if img_emb_local is not None:
            X_rank["img_emb"] = img_emb_local

        scores = model.predict(X_rank, batch_size=256).reshape(-1)

        tmp = pd.DataFrame({
            "user_id": df_eval["user_id"].values,
            "app_id": df_eval["app_id"].values,
            "label": df_eval["is_recommended"].astype(int).values,
            "score": scores,
        })
        if "genres" in df_eval.columns:
            tmp["genres"] = df_eval["genres"].values

        rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=["user_id", "app_id", "label", "score"])

    rank_df = pd.concat(rows, ignore_index=True)
    rank_df = rank_df.sort_values(["user_id", "score"], ascending=[True, False]).reset_index(drop=True)
    return rank_df

In [32]:
# Asumiendo que dense_features es una lista ej: ["price_final", "average_playtime", "rating"]
img_emb_zeros = np.zeros_like(img_emb, dtype=img_emb.dtype)
all_items = df_merged["app_id"].unique()

rank_df_pure_V2 = build_candidates_rank_df(
    model=model,
    test_df=test_merged,
    user_seen=user_seen,
    all_items=all_items,
    catalog_df=catalog_df,       # <--- Pasamos el catálogo
    id2idx=id2idx,
    text_emb=text_emb,
    feature_names=feature_names,
    appid2idx=appid2idx,
    img_emb=img_emb_zeros,
    dense_features=dense_features, # <--- Pasamos las features densas
    K=10,
    max_candidates=None          # <--- None para evaluar contra TODOS (sin atajos)
)

In [42]:
import numpy as np

def dcg_at_k(labels, k):
    labels = np.asarray(labels)[:k]
    if labels.size == 0:
        return 0.0
    gains = (2 ** labels - 1).astype(float)
    discounts = np.log2(np.arange(2, gains.size + 2))
    return float(np.sum(gains / discounts))

def ndcg_at_k(labels, k):
    labels = np.asarray(labels)
    ideal = np.sort(labels)[::-1]
    ideal_dcg = dcg_at_k(ideal, k)
    if ideal_dcg == 0.0:
        return 0.0
    return dcg_at_k(labels, k) / ideal_dcg

def average_precision_at_k(labels, k):
    labels = np.asarray(labels)[:k]
    if labels.size == 0:
        return 0.0
    hits = 0
    sum_prec = 0.0
    for i, y in enumerate(labels, start=1):
        if y == 1:
            hits += 1
            sum_prec += hits / i
    if hits == 0:
        return 0.0
    return sum_prec / min(hits, k)

def diversity_at_k(rank_df, k=10):
    divs = []

    for user_id, group in rank_df.groupby("user_id"):
        # Top-K por score
        topk = group.sort_values("score", ascending=False).head(k)

        if "genres" not in topk.columns:
            continue

        genres_list = topk["genres"].fillna("").astype(str).tolist()

        genres_split = set()
        for g in genres_list:
            for token in str(g).split(";"):
                token = token.strip()
                if token:
                    genres_split.add(token)

        divs.append(len(genres_split))

    return float(np.mean(divs)) if divs else 0.0

In [34]:
def macro_metrics_at_k(rank_df, k=10):
    users = rank_df["user_id"].unique()

    precs, recs, f1s, ndcgs = [], [], [], []
    hit_list, ap_list = [], []
    divs = []

    for u in users:
        df_u = rank_df[rank_df["user_id"] == u]
        if df_u.empty:
            continue

        topk = df_u.head(k)
        labels = topk["label"].astype(int).to_numpy()
        total_rel = int(df_u["label"].sum())

        hits_k = int(labels.sum())
        prec = hits_k / k if k > 0 else 0.0
        rec = hits_k / total_rel if total_rel > 0 else 0.0
        if prec + rec > 0:
            f1 = 2 * prec * rec / (prec + rec)
        else:
            f1 = 0.0

        ndcg = ndcg_at_k(labels, k)

        hit = 1.0 if hits_k > 0 else 0.0

        ap = average_precision_at_k(labels, k)

        # if "genres" in topk.columns:
        #     genres_list = topk["genres"].fillna("").astype(str).tolist()
        #     genres_split = set()
        #     for g in genres_list:
        #         for token in str(g).split(";"):
        #             token = token.strip()
        #             if token:
        #                 genres_split.add(token)
        #     div = len(genres_split) / min(k, len(topk)) if len(topk) > 0 else 0.0
        # else:
        #     div = np.nan
        
        if "genres" in topk.columns:
            genres_list = topk["genres"].fillna("").astype(str).tolist()
            genres_split = set()
            for g in genres_list:
                for token in str(g).split(";"):
                    token = token.strip()
                    if token:
                        genres_split.add(token)
            div = len(genres_split)  
        else:
            div = np.nan

        precs.append(prec)
        recs.append(rec)
        f1s.append(f1)
        ndcgs.append(ndcg)
        hit_list.append(hit)
        ap_list.append(ap)
        divs.append(div)

    metrics = {
        "Precision@K": float(np.nanmean(precs)) if precs else 0.0,
        "Recall@K": float(np.nanmean(recs)) if recs else 0.0,
        "F1@K": float(np.nanmean(f1s)) if f1s else 0.0,
        "NDCG@K": float(np.nanmean(ndcgs)) if ndcgs else 0.0,
        "HitRate@K": float(np.nanmean(hit_list)) if hit_list else 0.0,
        "MAP@K": float(np.nanmean(ap_list)) if ap_list else 0.0,
        "Diversity@K": float(np.nanmean(divs)) if divs else 0.0,
        
    }
    return metrics

## Caso V2 

In [35]:
metrics_k_v2 = macro_metrics_at_k(rank_df_pure_V2, k=10)
print(f"Resultados Top-10 (macro):")
for name, value in metrics_k_v2.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0010
Recall@K: 0.0089
F1@K: 0.0018
NDCG@K: 0.0036
HitRate@K: 0.0104
MAP@K: 0.0016
Diversity@K: 12.6117


In [43]:
diversity_k = diversity_at_k(rank_df_pure_V2, k=10)
print(f"Diversity@{10}: {diversity_k:.4f}")

Diversity@10: 12.6117


In [36]:


rank_df_pure_V2_imagen = build_candidates_rank_df(
    model=model,
    test_df=test_merged,
    user_seen=user_seen,
    all_items=all_items,
    catalog_df=catalog_df,       # <--- Pasamos el catálogo
    id2idx=id2idx,
    text_emb=text_emb,
    feature_names=feature_names,
    appid2idx=appid2idx,
    img_emb=img_emb_zeros,
    dense_features=dense_features, # <--- Pasamos las features densas
    K=10,
    max_candidates=None          # <--- None para evaluar contra TODOS (sin atajos)
)

In [37]:
metrics_k_v2_imagen = macro_metrics_at_k(rank_df_pure_V2_imagen, k=10)
print(f"Resultados Top-10 (macro):")
for name, value in metrics_k_v2_imagen.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0010
Recall@K: 0.0089
F1@K: 0.0018
NDCG@K: 0.0036
HitRate@K: 0.0104
MAP@K: 0.0016
Diversity@K: 12.6117


In [61]:
K = 10
metrics_k = macro_metrics_at_k(rank_df_pure, k=K)
print(f"Resultados Top-{K} (macro):")
for name, value in metrics_k.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 (macro):
Precision@K: 0.0141
Recall@K: 0.1094
F1@K: 0.0241
NDCG@K: 0.1168
HitRate@K: 0.1291
MAP@K: 0.1127
Diversity@K: 0.3673


In [ ]:
divs = metrics_k["Diversity@K"]  
print("Diversity@K (promedio de géneros distintos por usuario):", divs)

Diversity@K (promedio de géneros distintos por usuario): 0.3672505185526619


In [65]:
test_diversity =  float(np.nanmedian(divs)) if divs else 0.0
print("Diversity@K (mediana de géneros distintos por usuario):", test_diversity)

Diversity@K (mediana de géneros distintos por usuario): 0.3672505185526619


In [59]:
K = 10
recalls = []
for u in rank_df_pure["user_id"].unique():
    df_u = rank_df_pure[rank_df_pure["user_id"] == u]
    if df_u.empty:
        continue
    topk = df_u.head(K)
    labels = topk["label"].astype(int).to_numpy()
    total_rel = int(df_u["label"].sum())
    if total_rel == 0:
        continue
    rec = labels.sum() / total_rel
    recalls.append(rec)

import numpy as np
print("Recall@10 medio:", np.mean(recalls) if recalls else 0.0)
print("Recall@10 mediana:", np.median(recalls) if recalls else 0.0)
print("Usuarios con Recall@10 = 0:", (np.array(recalls) == 0).sum())

Recall@10 medio: 0.10939670686508532
Recall@10 mediana: 0.0
Usuarios con Recall@10 = 0: 7558


De todos los ítems realmente relevantes que hay en test (por usuario), el modelo solo recupera en promedio un 22 % dentro del Top‑10.
Visto por usuario: si un usuario tiene, por ejemplo, 1 ítem relevante, la probabilidad media de que aparezca en el Top‑10 es 0,22 (se acierta ese juego en unos 22 de cada 100 usuarios).

Sin embedding de imagen 

In [ ]:
img_emb_zeros = np.zeros_like(img_emb, dtype=img_emb.dtype)

rank_df_sin_img_all = build_candidates_rank_df(
    model=model,
    test_df=test_merged,
    user_seen=user_seen,
    all_items=all_items,
    id2idx=id2idx,
    text_emb=text_emb,
    feature_names=feature_names,
    appid2idx=appid2idx,
    img_emb=img_emb_zeros,   
    K=10,
    max_candidates=None,
)

metrics_no_image_all = macro_metrics_at_k(rank_df_sin_img_all, k=10)
print("Resultados Top-10 sin imagen:")
for name, value in metrics_no_image_all.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 sin imagen:
Precision@K: 0.0814
Recall@K: 0.6655
F1@K: 0.1413
NDCG@K: 0.6937
HitRate@K: 0.6957
MAP@K: 0.6929
Diversity@K: 0.8015


In [ ]:
top10 = rank_df_pure.groupby("user_id").head(10)

def count_genres(s):
    if pd.isna(s) or str(s).strip() == "":
        return 0
    return len({t.strip() for t in str(s).split(";") if t.strip()})

top10["num_genres_item"] = top10["genres"].apply(count_genres)
print("Media de géneros por ítem en Top10:", top10["num_genres_item"].mean())

divs_por_usuario = []
for u, df_u in top10.groupby("user_id"):
    genres_split = set()
    for g in df_u["genres"].fillna("").astype(str):
        for token in g.split(";"):
            token = token.strip()
            if token:
                genres_split.add(token)
    divs_por_usuario.append(len(genres_split))

import numpy as np
print("Media de géneros distintos en Top10:", np.mean(divs_por_usuario))
print("Mediana de géneros distintos en Top10:", np.median(divs_por_usuario))

C:\Users\Casa\AppData\Local\Temp\ipykernel_12592\2204716964.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top10["num_genres_item"] = top10["genres"].apply(count_genres)


Media de géneros por ítem en Top10: 0.03862641161557963
Media de géneros distintos en Top10: 0.3672505185526619
Mediana de géneros distintos en Top10: 0.0


## Cambio el max candidates

In [49]:
rank_df_pure_200 = build_candidates_rank_df(
    model=model,
    test_df=test_merged,
    user_seen=user_seen,
    all_items=all_items,
    id2idx=id2idx,
    text_emb=text_emb,
    feature_names=feature_names,
    K=10,
    max_candidates=200,
    appid2idx=appid2idx,
    img_emb=img_emb,
)

metrics_200 = macro_metrics_at_k(rank_df_pure_200, k=10)
print("Resultados Top-10 con max_candidates=200:")
for name, value in metrics_200.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 con max_candidates=200:
Precision@K: 0.0395
Recall@K: 0.3096
F1@K: 0.0678
NDCG@K: 0.2311
HitRate@K: 0.3384
MAP@K: 0.1971
Diversity@K: 0.0665


Sin embedding de imagenes y con max_candidates

In [ ]:
img_emb_zeros = np.zeros_like(img_emb, dtype=img_emb.dtype)

rank_df_sin_img = build_candidates_rank_df(
    model=model,
    test_df=test_merged,
    user_seen=user_seen,
    all_items=all_items,
    id2idx=id2idx,
    text_emb=text_emb,
    feature_names=feature_names,
    appid2idx=appid2idx,
    img_emb=img_emb_zeros,   
    K=10,
    max_candidates=200,
)

metrics_no_image = macro_metrics_at_k(rank_df_sin_img, k=10)
print("Resultados Top-10 con max_candidates=200:")
for name, value in metrics_no_image.items():
    print(f"{name}: {value:.4f}")

Resultados Top-10 con max_candidates=200:
Precision@K: 0.0815
Recall@K: 0.6655
F1@K: 0.1413
NDCG@K: 0.6956
HitRate@K: 0.6957
MAP@K: 0.6956
Diversity@K: 0.0773


In [ ]:
metrics_no_image = macro_metrics_at_k(rank_df_sin_img, k=10)
print("Resultados Top-10 con max_candidates=200:")
for name, value in metrics_no_image.items():
    print(f"{name}: {value:.4f}")

## Modelo solo texto

In [47]:
'''
only_text_features = [DenseFeat("text_emb", emb_dim)]

model_text = DeepFM(
    linear_feature_columns=only_text_features,
    dnn_feature_columns=only_text_features,
    task="binary",
    device=device,
)

model_text.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])

model_text.fit(
    {"text_emb": train_text_emb},
    y_train,
    batch_size=256,
    epochs=5,
    verbose=2,
    validation_data=({"text_emb": val_text_emb}, y_val),
)
'''

'\nonly_text_features = [DenseFeat("text_emb", emb_dim)]\n\nmodel_text = DeepFM(\n    linear_feature_columns=only_text_features,\n    dnn_feature_columns=only_text_features,\n    task="binary",\n    device=device,\n)\n\nmodel_text.compile(optimizer="adam", loss="binary_crossentropy", metrics=["auc"])\n\nmodel_text.fit(\n    {"text_emb": train_text_emb},\n    y_train,\n    batch_size=256,\n    epochs=5,\n    verbose=2,\n    validation_data=({"text_emb": val_text_emb}, y_val),\n)\n'

# Interpretacion de Resultados

**Análisis de los resultados del modelo**

**Equilibrio de clases y estructura de los datos**

En los tres conjuntos (train, validación y test) el porcentaje de reseñas positivas es muy alto (alrededor de 86%). Además, en test la gran mayoría de usuarios tiene como máximo una reseña positiva: la media de positivos por usuario es 0,86, la mediana y el tercer cuartil son 1, y solo una fracción de usuarios no tiene ningún positivo. Esto implica que, por construcción del dataset, el modelo debe encontrar como mucho un ítem relevante por usuario entre muchos candidatos.

**Calidad del modelo como clasificador**

En términos de clasificación binaria, el modelo DeepFM obtiene un AUC en test ≈ 0,70 y un LogLoss ≈ 0,57. Estos valores indican que el modelo es capaz de distinguir razonablemente bien entre reseñas positivas y negativas: está claramente por encima del azar (AUC 0,5) y aprende patrones útiles tanto de las características tabulares (user_id, rating, precio, tiempo de juego) como del embedding de texto. Es decir, como modelo de predicción de probabilidad de “is_recommended”, el desempeño es aceptable dadas las limitaciones del problema. 

max_candidates controla cuántos ítems negativos se usan como candidatos por usuario para hacer el ranking, más los positivos obligatorios. Sirve para limitar el tamaño del conjunto de candidatos y que la evaluación sea computacionalmente manejable sin cambiar demasiado las métricas. Si este fuese max_candidate=None, pra cada usuario se tomarían todos los candidatos posibles: candidates = todos los all_items que no están en seen_u. Entonces items_eval = pos_items ∪ sampled_neg incluiría todos los negativos posibles.

**Interpretación de las métricas Top‑N**

En el escenario de recomendación Top‑10 con exclusión de ítems vistos en entrenamiento y validación, el modelo obtiene:

Precision@10 ≈ 0,0565
Recall@10 ≈ 0,5654
HitRate@10 ≈ 0,5654
NDCG@10 ≈ 0,5654
MAP@10 ≈ 0,5654
Diversity@10 ≈ 0,0377


Como cada usuario tiene como máximo un ítem positivo en test, Recall@10, HitRate@10, NDCG@10 y MAP@10 prácticamente coinciden: todas miden, en la práctica, la probabilidad de que ese único ítem relevante aparezca dentro del Top‑10. Un valor alrededor de 0,56 significa que el modelo consigue recomendar el juego realmente relevante en el Top‑10 para aproximadamente el 56 % de los usuarios, mientras que para el 44 % restante no lo consigue colocar entre las primeras diez posiciones. Además, al analizar la distribución por usuario, el Recall@10 medio es ~0,56 pero la mediana es 1,0, lo que indica que más de la mitad de los usuarios recuperan su ítem relevante en el Top‑10, y el grupo de usuarios donde el modelo no acierta (Recall@10 = 0) es el que arrastra hacia abajo la media.

**Comportamiento de la precisión y la diversidad**

La Precision@10 es baja en términos absolutos (0,056) porque el denominador (10 recomendaciones por usuario) es grande en relación al máximo número de positivos por usuario (1). Incluso cuando el modelo acierta, solo 1 de los 10 ítems es verdaderamente relevante, de modo que la precisión máxima posible en este setup es 0,1. Por otra parte, la Diversity@10 es muy baja (0,038), lo que sugiere que las recomendaciones tienden a concentrarse en pocos géneros: el modelo no solo se centra en un subconjunto de ítems, sino también en un subconjunto reducido de tipos de juegos, reduciendo la variedad en el Top‑10.

**Efecto del tamaño del conjunto de candidatos**

Al comparar las métricas con distintos tamaños del conjunto de candidatos (por ejemplo, max_candidates = 500 frente a max_candidates = 200), los resultados cambian. Las metricas mejoran ya que se esta reduciendo la cantidad de negativos, donde en vez de comparar el juego relevante contra, todos los juegos no relevantes,
solo se comparas contra una muestra más pequeña, por ejemplo 200 juegos no relevantes. Eso hace el ranking más fácil, porque hay menos juegos “malos” compitiendo por entrar al Top‑10.

**Conclusión general**

En conjunto, los resultados muestran que el modelo DeepFM aprende patrones útiles (AUC ≈ 0,70) y que, en un escenario de recomendación Top‑10 realista con un solo positivo por usuario, es capaz de recuperar el juego relevante para algo más de la mitad de los usuarios. Sin embargo, sigue habiendo un porcentaje significativo de usuarios para los que las recomendaciones no incluyen el ítem relevante, y la diversidad de los juegos recomendados es baja. Estas limitaciones no se deben solo al modelo, sino también a la naturaleza del problema (datos muy sesgados hacia reseñas positivas, máximo un positivo por usuario y un conjunto de candidatos grande), lo que hace que las métricas Top‑N resulten exigentes y difíciles de mejorar de forma drástica sin cambiar la definición del escenario o enriquecer aún más las características disponibles.



# Escenario de ejecución

**Escenario de evaluación y dificultad del problema**

Para evaluar el modelo se adoptó un escenario de recomendación intencionadamente exigente y cercano a un caso de uso real. En particular, al generar el Top‑10 de juegos recomendados para cada usuario se excluyen todos los ítems que ese usuario ya ha visto en los conjuntos de entrenamiento y validación. De este modo, el sistema solo puede recomendar juegos que son “nuevos” para el usuario, simulando un contexto de descubrimiento de contenido en producción, donde el objetivo es proponer títulos que el usuario aún no ha consumido.

Este planteamiento tiene dos consecuencias importantes. En primer lugar, en el conjunto de test la gran mayoría de los usuarios dispone de como máximo una reseña positiva: la media de positivos por usuario es 0,86, la mediana es 1 y solo una fracción minoritaria no tiene ningún positivo. En la práctica, esto significa que el modelo debe localizar un único juego relevante por usuario dentro de un catálogo amplio de candidatos, sin apoyo en los juegos ya conocidos. En segundo lugar, esta configuración vuelve las métricas Top‑N especialmente estrictas: valores como Recall@10, HitRate@10, NDCG@10 o MAP@10 pasan a medir esencialmente la probabilidad de que ese único ítem relevante aparezca dentro de las diez primeras posiciones, por lo que resulta difícil obtener valores cercanos a 1 incluso con un modelo razonablemente bueno.

En este contexto, los resultados obtenidos (recuperar el ítem relevante en el Top‑10 para aproximadamente el 56 % de los usuarios, con un AUC en test cercano a 0,70) deben interpretarse teniendo en cuenta la dureza del escenario. Aunque las métricas Top‑10 no alcanzan valores muy altos en términos absolutos, reflejan el comportamiento del modelo en un entorno de recomendación realista, con exclusión de ítems vistos y un catálogo grande, lo que justifica que no sean directamente comparables con trabajos que utilizan configuraciones de evaluación más sencillas o menos restrictivas.

**Embeddings de texto**

Para representar el contenido textual de cada juego se construyó primero una columna text_input combinando la descripción larga (about_the_game) y el título del juego (game_title), rellenando valores faltantes con cadena vacía. Sobre estos textos se generaron embeddings mediante la librería sentence_transformers, utilizando el modelo preentrenado all-MiniLM-L6-v2. Este modelo es un Transformer tipo BERT reducido (MiniLM), especializado en producir sentence embeddings densos de dimensión 384 que capturan la semántica global de la frase o documento. En la fase de preprocesamiento, todos los textos de sampled_recs_merged se codificaron con este modelo, obteniendo una matriz text_emb de tamaño (N,384) que se guardó en disco (text_emb_sampled.npy). Posteriormente, dichos vectores se cargaron, se alinearon con cada reseña a través del identificador review_id para los subconjuntos de entrenamiento, validación y test, y finalmente se incorporaron al modelo DeepFM como una característica densa adicional (DenseFeat("text_emb", emb_dim)), permitiendo que la red aprenda a combinar la información semántica del texto con las variables tabulares del usuario y del juego.

# Resumen de tres efectos principales sobre métricas y aprendizaje:



**1. Cada usuario aporta como máximo 1 “objeto relevante”**

- Si casi todos los usuarios tienen solo 1 reseña positiva en test, entonces por usuario solo existe un único ítem que realmente importa.

- En Top‑10, el mejor caso por usuario es: 1 acierto y 9 errores. Eso fija un techo:

i) Precisión máxima por usuario: 1/10=0,1.

ii) Recall@10 pasa a ser simplemente: ¿lo pusiste o no lo pusiste en el Top‑10? (0 o 1).

**2. Recall, HitRate, NDCG y MAP colapsan a la misma idea**

Con un solo positivo por usuario:
- Recall@10 = 1 si el ítem relevante está en Top‑10, 0 en caso contrario.
- HitRate@10 = lo mismo (1 si hay al menos un acierto).
- NDCG@10 y MAP@10 también se reducen básicamente a “si el ítem relevante aparece en las primeras posiciones o no”.
Por eso en tus resultados estas métricas salen casi iguales (~0.56): todas están midiendo la probabilidad de “acertar” ese único ítem.

**3. La evaluación es muy dura y limita cuánto puedes mejorar**

- Imagina que por usuario tuvieras 5 ítems positivos en test: aunque aciertes solo 2, ya tendrías Recall@10 = 2/5 = 0.4, y podrías subir más si el modelo coloca varios positivos en el Top‑10.
- Con 1 positivo por usuario, o aciertas (1) o fallas (0); no hay término medio.
Eso hace que:

i) La métrica global sea muy sensible a usuarios donde el modelo falla.

ii) Sea difícil llegar a valores cercanos a 1 aunque el modelo sea razonablemente bueno: necesitas acertar el único ítem relevante para casi todos los usuarios.

En resumen: que la mayoría de usuarios tenga solo una reseña positiva convierte el problema de ranking en un juego de “adivinar un único ítem correcto entre muchos”, con precisión máxima muy baja y métricas Top‑N muy estrictas; por eso tus ~0.56 de Recall@10/HitRate/… son más razonables de lo que parecen a primera vista.

In [ ]:
user_counts = test_merged.groupby("user_id")["review_id"].count()

num_users_10plus = (user_counts >= 10).sum()
print("Usuarios con >=10 reseñas en test:", num_users_10plus)

pct_users_10plus = (user_counts >= 10).mean()
print("Porcentaje de usuarios con >=10 reseñas:", pct_users_10plus)

Usuarios con >=10 reseñas en test: 16
Porcentaje de usuarios con >=10 reseñas: 0.0016151827175449223


In [ ]:
import pandas as pd

col = df_merged["genres"].dropna().astype(str)

all_genres = set()
for g in col:
    for token in g.split(";"):
        token = token.strip()
        if token:
            all_genres.add(token)

print("Número de géneros distintos:", len(all_genres))
print("Algunos ejemplos:", list(all_genres)[:20])

Número de géneros distintos: 27
Algunos ejemplos: ['Simulation', 'Web Publishing', 'Sports', 'Video Production', 'Indie', 'Casual', 'RPG', 'Nudity', 'Software Training', 'Racing', 'Game Development', 'Strategy', 'Design & Illustration', 'Photo Editing', 'Sexual Content', 'Audio Production', 'Animation & Modeling', 'Gore', 'Education', 'Free to Play']
